# `gtmf_creation.py` on Colab: one Structured Clinical Reference per admission

This notebook runs the extraction half of `gtmf_creation.py` -- `extract_gtmf`, the JSON repair
helpers, the light-case filter and the `process_notes` batch loop -- inside a Colab runtime, with the
extractor served on the runtime's GPU.

It **imports** those functions from the repository rather than copying them. There is one definition
of the prompt, the evidence-offset repair and the failure behaviour, and this notebook is a driver
for it; a copy would drift and silently stop being the thing the thesis reports on.

**Prerequisites -- all of them, or the run stops partway:**

1. A GPU runtime: *Runtime -> Change runtime type -> T4 / L4 / A100*.
2. PhysioNet credentialing for MIMIC-III, with **BigQuery access approved for the same Google
   account** you sign in with below (PhysioNet -> account settings -> Cloud). Skip if you are
   pointing this at a local CSV directory instead.
3. Your own Google Cloud project. BigQuery bills the reader, not the publisher.

**What leaves the runtime: nothing.** The model server runs inside the runtime and is reached over
loopback. The provider layer checks that *before* it opens a socket, so restricted clinical text
cannot reach a hosted API (decision D2 / GOV-3). Reading MIMIC-III *from* BigQuery is the same
direction as downloading the CSVs and does not touch that rule.

**What this notebook does not do: choose cases by reading note text.** `gtmf_creation.main()` was
withdrawn for exactly that -- it ranked notes by `is_light_common_case`, a keyword scan over the note
body, and a cohort chosen from the text it is then measured against cannot reproduce by hash (M3).
Cases here come from a `meddial-cohort` manifest, or, for exploration only, from the structured
tables in a fixed order. `is_light_common_case` still appears below, where it belongs: as the gate
`process_notes` applies, reported before you run it so you can see what it will drop.

Everything written under `/content` is derived from restricted data and dies with the runtime.
Section 9 packages it; where it may then be stored is a governance decision, so this notebook does
not mount Drive or upload anything for you.

## 0. Confirm the GPU

In [ ]:
!nvidia-smi

# No output above means the runtime has no GPU: Runtime -> Change runtime type -> T4 / L4 / A100.
# Extraction still runs on CPU, at roughly an order of magnitude less throughput.

## 1. Serve the extractor on the runtime's GPU

Ollama is installed into the runtime and left listening on `localhost:11434`. That is the only shape
of model server this pipeline accepts for MIMIC-derived text.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time
import httpx

subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        httpx.get('http://localhost:11434/api/tags', timeout=2.0).raise_for_status()
        print('ollama is serving')
        break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError('ollama did not come up; re-run this cell')

### Pick the largest extractor the card actually holds

Implementation Plan §12.4 asks for the largest extractor the hardware allows, because extraction
error propagates into every downstream metric. What it must not do is exceed VRAM: a model that
swaps is not a slow run, it is a stalled one.

The tag is exported as `MEDDIAL_GTMF_MODEL`, which is the variable `gtmf_creation.provider_from_env`
reads, so the same choice drives both this notebook and that helper.

In [ ]:
# torch is preinstalled on Colab and is not a dependency of this package, so a
# runtime without it is assumed to have no GPU rather than failing here.
try:
    import torch
    gib = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
except ImportError:
    gib = 0

# Weights at Q4 plus KV cache, with headroom. Raise these only after watching nvidia-smi.
if gib >= 38:    EXTRACTOR = 'qwen2.5:32b'   # A100 40GB
elif gib >= 21:  EXTRACTOR = 'qwen2.5:14b'   # L4 24GB
elif gib >= 14:  EXTRACTOR = 'qwen2.5:14b'   # T4 16GB, tight but fits at Q4
else:            EXTRACTOR = 'qwen2.5:7b'    # CPU or a small card

os.environ['MEDDIAL_GTMF_MODEL'] = EXTRACTOR
os.environ['MEDDIAL_GTMF_BASE_URL'] = 'http://localhost:11434/v1'
print(f'{gib:.0f} GiB of VRAM -> {EXTRACTOR}')

In [ ]:
!ollama pull $MEDDIAL_GTMF_MODEL

# The whole note goes in one call, so the serving context has to hold it. The longest concatenated
# discharge documentation in the 1,000-case cohort is ~27.7k characters (~7k tokens); with the
# schema and a 4,096-token answer that is ~12.5k. Ollama's default 4k window would truncate, and a
# truncated answer is unparseable -- extract_gtmf raises ExtractionError rather than returning an
# empty reference, so a short window shows up as every case failing.
!printf 'FROM %s\nPARAMETER num_ctx 16384\n' "$MEDDIAL_GTMF_MODEL" > /content/Modelfile
!ollama create meddial-extractor -f /content/Modelfile

In [ ]:
# Use the 16k-context copy for extraction.
os.environ['MEDDIAL_GTMF_MODEL'] = 'meddial-extractor'
print(os.environ['MEDDIAL_GTMF_MODEL'])

## 2. Install MedDial

The `bigquery` extra adds the BigQuery client; without it the CSV path still works.

The install is thin on purpose -- Pydantic, dotenv, httpx, pandas and the BigQuery client. The
scoring stack (`eval`) is left out because nothing on this path imports it, and because installing it
would drag `huggingface-hub` back to 0.x to satisfy `transformers` 4.x, breaking the `gradio` and
`diffusers` this image ships. Extraction needs none of it.

In [ ]:
!git clone -q https://github.com/alongott15/FinalProject-MedDial.git /content/FinalProject-MedDial
%pip install -q -e '/content/FinalProject-MedDial[bigquery]'

In [ ]:
import os, sys

REPO = '/content/FinalProject-MedDial'
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

from gtmf_creation import (
    ExtractionError,
    extract_gtmf,
    get_existing_gtmf_ids,
    is_light_common_case,
    process_notes,
)

print('gtmf_creation imported from', REPO)

## 3. Where MIMIC-III comes from

`MimicSource` is the single place that decides what a candidate admission is; the two backends only
differ in how the six tables are fetched. Set `SOURCE` to `'bigquery'` (Colab's usual case -- the CSV
distribution is tens of gigabytes) or `'csv'` if you have mounted an extract yourself.

In [ ]:
SOURCE = 'bigquery'          # 'bigquery' or 'csv'
BQ_PROJECT = 'your-gcp-project-id'   # <- edit: your own project; BigQuery bills the reader
CSV_DIR = '/content/drive/MyDrive/mimic-iii'   # only used when SOURCE == 'csv'

if SOURCE == 'bigquery':
    from google.colab import auth
    auth.authenticate_user()
    os.environ['MIMIC_BIGQUERY_PROJECT'] = BQ_PROJECT

    from meddial.cohort.mimic_bigquery import MimicBigQuerySource
    source = MimicBigQuerySource(BQ_PROJECT)
else:
    from meddial.cohort.mimic_csv import MimicCsvSource
    source = MimicCsvSource(CSV_DIR)

print(type(source).__name__)

In [ ]:
# Pre-flight. This fails in seconds if PhysioNet credentialing is not linked to the account you just
# signed in with -- far better than failing an hour into extraction.
snapshot = source.snapshot_hash()
print('source snapshot:', snapshot)

## 4. Which admissions to extract

Two ways, and they are not equivalent.

**A cohort manifest (use this for anything reported).** `meddial-cohort` applies E1-E10 to the
structured fields and writes `cohort_private_manifest.json`, along with the exclusion count at each
criterion. Point `COHORT_MANIFEST` at one -- built in this runtime or copied in -- and only those
admissions are read. The manifest also records the snapshot hash it was selected from, and the cell
below refuses a mismatch for the same reason `meddial-scr` does: a CSV hash and a BigQuery hash never
compare equal, so this catches selecting from one backend and extracting from the other.

**No manifest (exploration only).** The first `LIMIT` admissions in `(subject_id, admittime, hadm_id)`
order that have discharge documentation. Deterministic and structural -- no note vocabulary is read
to choose them -- but it is not a cohort and nothing selected this way belongs in a results table.

To build a manifest here first, run:

```
!meddial-cohort --bigquery --out /content/meddial-out/cohort --n 200
```

In [ ]:
COHORT_MANIFEST = None   # e.g. '/content/meddial-out/cohort/cohort_private_manifest.json'
LIMIT = 5                # cases to extract this pass; None for all of them

import json

wanted = None
if COHORT_MANIFEST:
    manifest = json.loads(open(COHORT_MANIFEST, encoding='utf-8').read())
    if manifest.get('source_snapshot_hash') != snapshot:
        raise SystemExit(
            'Refusing to extract: this source is not the one the cohort was selected from.\n'
            f"  manifest:    {manifest.get('source_snapshot_hash')}\n"
            f"  this source: {snapshot}"
        )
    wanted = {(int(c['subject_id']), int(c['hadm_id'])) for c in manifest['selected']}
    print(f"cohort: {len(wanted)} case(s), cohort_hash={manifest.get('cohort_hash')}")
else:
    print('no manifest: exploration mode, results from this pass are not a cohort')

### Build the rows `process_notes` expects

`process_notes` takes plain dicts -- the shape `Utils.csv_data_loader` used to hand it. They are
assembled here from `MimicSource` instead, so the note text is the admission's *whole* discharge
documentation (summary plus addenda, in filing order) rather than whichever row happened to rank
first. On a 1,000-case cohort, taking the top-ranked row alone read a note that was not the longest
available in 6.5% of admissions, losing a median of 6,132 characters.

One caveat worth seeing before it lands in a file: `process_notes` writes `Date_of_Birth` into every
reference, and MIMIC shifts the DOB of patients over 89, so those ages compute to roughly 300.
Criterion E5 excludes them, so a manifest-driven run will not contain any; an exploration run can.
`meddial-scr` omits DOB entirely -- an age is what the disclosure policies act on, and a date of
birth is a re-identification vector in a document that outlives the extract.

The first pass over NOTEEVENTS is the expensive part (a few GB on BigQuery, streamed). It happens
once, here.

In [ ]:
import pandas as pd

patients = source.table('PATIENTS', parse_dates=['DOB'])
dob_by_subject = {
    int(r.SUBJECT_ID): r.DOB
    for r in patients.itertuples(index=False)
    if not pd.isna(r.SUBJECT_ID)
}
demographics = source.demographics()

rows = []
for record in source.admission_records():
    key = (record.subject_id, record.hadm_id)
    if wanted is not None and key not in wanted:
        continue
    if not record.note_text:
        continue
    extra = demographics.get(key, {})
    rows.append({
        'row_id': record.row_id or 0,
        'subject_id': record.subject_id,
        'hadm_id': record.hadm_id,
        'text': record.note_text,
        'category': record.note_category,
        'dob': dob_by_subject.get(record.subject_id),
        'admittime': record.admittime,
        'dischtime': record.dischtime,
        'gender': extra.get('Sex', ''),
        'religion': extra.get('Religion', ''),
        'marital_status': extra.get('Marital_Status', ''),
        'ethnicity': extra.get('Ethnicity', ''),
        'insurance': extra.get('Insurance', ''),
        'admission_type': extra.get('Admission_Type', ''),
    })

rows.sort(key=lambda r: (r['subject_id'], r['hadm_id']))
if LIMIT is not None:
    rows = rows[:LIMIT]

print(f'{len(rows)} admission(s) ready')
if rows:
    lengths = sorted(len(r['text']) for r in rows)
    print(f'note length: min {lengths[0]}, median {lengths[len(lengths)//2]}, max {lengths[-1]} chars')

## 5. The provider

`gtmf_creation.provider_from_env()` builds this from `MEDDIAL_GTMF_BASE_URL` / `MEDDIAL_GTMF_MODEL`
in one line, and it is the right thing at a prompt. It is built out here because two of its defaults
are wrong for a whole-note extraction on a Colab card:

* **timeout.** The provider's 300 s default is a chat timeout. A whole-note extraction exceeds it,
  and because a timeout is retried, each case then burns three full timeouts and produces nothing.
  `meddial-scr` uses 1800 s; so does this.
* **reasoning budget.** Extraction emits a fixed schema, so reasoning tokens are cost without
  benefit -- and on a reasoning model they consume the whole `max_tokens` budget and leave the
  message empty. `'none'` turns that off where the server supports it.

The digest, not the tag, is what a result is attributed to: tags are mutable, so a run pinned only by
name cannot be reproduced (C8). `resolve_ollama_digest` reads it off the server and refuses to
continue without one.

In [ ]:
from meddial.llm import LocalOpenAICompatibleProvider, resolve_ollama_digest

BASE_URL = os.environ['MEDDIAL_GTMF_BASE_URL']
MODEL = os.environ['MEDDIAL_GTMF_MODEL']

provider = LocalOpenAICompatibleProvider(
    BASE_URL,
    MODEL,
    model_digest=resolve_ollama_digest(BASE_URL, MODEL),
    model_family=os.environ.get('MEDDIAL_GTMF_FAMILY', EXTRACTOR.split(':')[0]),
    quantisation=os.environ.get('MEDDIAL_GTMF_QUANT', 'Q4_K_M'),
    reasoning_effort='none',
    timeout_s=1800.0,
)
print('provider ready:', MODEL)

## 6. What the light-case gate will do to this batch

`process_notes` runs `is_light_common_case` over each note and skips the ones it rejects: an ICU or
severity term anywhere in the text excludes the case, and at least one light symptom term must
appear. It is a keyword scan, so it reads the note body -- which is why it may gate *what gets
extracted* but must never choose *which cases exist*. Run it first and look at the numbers: on a
manifest-driven batch it will usually reject most admissions, and those produce no file at all.

If you want a reference for every case the manifest names, use `meddial-scr`, which has no such gate.

In [ ]:
from collections import Counter

verdicts = [(r['subject_id'], r['hadm_id'], is_light_common_case(r['text'])) for r in rows]
passed = [v for v in verdicts if v[2]['passed']]

print(f'{len(passed)}/{len(verdicts)} would pass the light-case gate\n')
for reason, n in Counter(v[2]['reason'].split(':')[0] for v in verdicts if not v[2]['passed']).most_common():
    print(f'  {n:>4}  rejected: {reason}')

for subject_id, hadm_id, verdict in verdicts[:5]:
    print(f"\n{subject_id}_{hadm_id}: {'PASS' if verdict['passed'] else 'SKIP'} -- {verdict['reason'][:160]}")

## 7. Extract one note first

A single call, watched, before committing an hour to the batch. This is the whole of `extract_gtmf`:
one call per note, no chunking. Splitting cost more than it bought -- the ~1,200-token schema was
resent with every piece, an entity described across a boundary was seen whole by neither call, and
merging made the result depend on where the boundary fell.

The two reports underneath are the ones that matter. `evidence_issues` lists spans that do not
resolve against the note; `unevidenced_entities` lists entities that carry no span at all. Both are
reported rather than raised, because extraction recall is itself a measured quantity (GRND-1/2) --
an unevidenced entity is a finding, not a crash. Offsets are repaired automatically from the quoted
text before this point, so what remains here is a quote that is not in the note.

If this raises `ExtractionError`, the answer was truncated or unparseable: raise `max_tokens`, or
widen `num_ctx` in section 1.

In [ ]:
from IPython.display import Markdown, display
from Utils.markdown_gtmf import gtmf_to_markdown

probe = rows[0]
note_id = f"{probe['subject_id']}_{probe['hadm_id']}"
print(f"{note_id}: {len(probe['text'])} chars")

reference = extract_gtmf(probe['text'], provider, note_id=note_id, max_tokens=4096)

print(f'\nsymptoms:   {len(reference.core.symptoms)}')
print(f'diagnoses:  {len(reference.core.diagnoses)}')
print(f'treatments: {len(reference.core.treatments)}')

issues = reference.evidence_issues({note_id: probe['text']})
print(f'\nunresolved evidence spans: {len(issues)}')
for path, reason in list(issues.items())[:10]:
    print(f'  {path}: {reason}')

unevidenced = reference.unevidenced_entities({note_id: probe['text']})
print(f'entities with no resolvable evidence: {len(unevidenced)}')
for path in unevidenced[:10]:
    print(f'  {path}')

In [ ]:
display(Markdown(gtmf_to_markdown(reference)))

## 8. Run the batch

`process_notes` writes `gtmf_<subject_id>_<hadm_id>.md` per admission and returns the quality
summary. It is **resumable**: `get_existing_gtmf_ids` reads the output directory first and skips
every case already written, which is what makes a recycled Colab runtime survivable -- re-run this
cell and it continues, provided the directory is still there. If it is not, copy it off first
(section 9).

`gtmfs_created + light_case_failed + skipped_existing` should account for every row; anything else is
in `json_parse_failures` or was logged as an error above.

In [ ]:
OUTPUT_DIR = '/content/meddial-out/gtmf'

print(f'{len(get_existing_gtmf_ids(OUTPUT_DIR))} profile(s) already present -- those are skipped\n')

summary = process_notes(rows, provider, output_dir=OUTPUT_DIR)

print()
for key, value in summary.items():
    print(f'  {key:>20}: {value}')

## 9. Inspect what was written

In [ ]:
from pathlib import Path
from Utils.markdown_gtmf import load_gtmf_markdown

written = sorted(Path(OUTPUT_DIR).glob('gtmf_*.md'))
print(f'{len(written)} file(s) in {OUTPUT_DIR}')
for path in written[:20]:
    print(f'  {path.name}  ({path.stat().st_size} bytes)')

In [ ]:
if written:
    loaded = load_gtmf_markdown(str(written[0]))
    print(json.dumps(loaded, indent=2, ensure_ascii=False)[:3000])

## 10. Take the output with you

Everything under `/content/meddial-out` is derived from restricted data and disappears with the
runtime. Where it may be stored is a governance decision, not a convenience one, so this cell only
packages the directory. Run it, then move the archive somewhere your data use agreement covers.

In [ ]:
!cd /content && zip -qr meddial-gtmf.zip meddial-out && ls -lh /content/meddial-gtmf.zip

# from google.colab import files; files.download('/content/meddial-gtmf.zip')